In [1]:
from huggingface_hub import login
login()


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [3]:
# Looking at what parts of the dataset are available
from datasets import get_dataset_config_names
print(get_dataset_config_names("Exploration-Lab/CS-781-Capstone"))


README.md:   0%|          | 0.00/16.1k [00:00<?, ?B/s]

['phase_1', 'phase_2', 'phase_3', 'phase_4', 'phase_5', 'phase_6', 'phase_7']


In [5]:
from huggingface_hub import list_repo_files
files = list_repo_files("Exploration-Lab/CS-781-Capstone", repo_type="dataset")
print([f for f in files if "lner" in f.lower()])


['Phase_2_LNER/Test/fold3-text.json', 'Phase_2_LNER/Train/fold1-gold.json', 'Phase_2_LNER/Train/fold1-text.json', 'Phase_2_LNER/Train/fold2-gold.json', 'Phase_2_LNER/Train/fold2-text.json', 'Phase_2_LNER/Train/labels.txt']


In [6]:
# Checking the real labels and how many documents each fold has
from huggingface_hub import hf_hub_download
import json

text1_path = hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_2_LNER/Train/fold1-text.json", repo_type="dataset")
gold1_path = hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_2_LNER/Train/fold1-gold.json", repo_type="dataset")
labels_path = hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_2_LNER/Train/labels.txt", repo_type="dataset")

with open(text1_path) as f:
    text1 = json.load(f)

with open(gold1_path) as f:
    gold1 = json.load(f)

with open(labels_path) as f:
    labels = f.read()

print(labels)
print(len(text1), "documents in fold1 text")
print(len(gold1), "documents in fold1 gold")


APP
RESP
A.COUNSEL
R.COUNSEL
JUDGE
WIT
AUTH
COURT
STAT
PREC
DATE
CASENO
35 documents in fold1 text
35 documents in fold1 gold


In [7]:
# Picking one document and looking at its real text and gold spans together
doc_id = list(text1.keys())[0]
print("doc_id:", doc_id)
print("text length:", len(text1[doc_id]))
print(text1[doc_id][:500])
print()
print("gold spans for this doc:")
print(gold1[doc_id])


doc_id: 115651329
text length: 18616
REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 

gold spans for this doc:
[[137, 153, 'RESP', 'STATE OF HARYANA'], [252, 261, 'DATE', '19.8.2011'], [276, 308, 'COURT', 'High Court of Punjab and Haryana'], [359, 369, 'COURT', 'High Court'], [575, 585, 'DATE', '18.11.1994'], [637, 644, 'WIT', 'Jagdish'], [695, 704, 'WIT', 'Mange Ram'], [792, 800, 'APP', 'Jage Ram'], [1122, 1130, 'APP', 'Jage Ram'], [1341, 1349, 'APP', 'Jage Ram'], [1373, 1380, 'WIT', 'Jagdish'], [1583, 1590, 'WIT', 'Jagdish'], [1809, 1822, 

In [8]:
# Checking that every gold span's offsets actually match its given entity text
mismatches = 0
for start, end, label, entity_text in gold1[doc_id]:
    if text1[doc_id][start:end] != entity_text:
        mismatches += 1
        print(start, end, label, repr(text1[doc_id][start:end]), "vs", repr(entity_text))

print("total spans:", len(gold1[doc_id]))
print("mismatches:", mismatches)


total spans: 104
mismatches: 0


In [9]:
# Checking offset correctness across every document in both folds, not just one
def check_fold(text_data, gold_data):
    total = 0
    bad = 0
    for doc_id, spans in gold_data.items():
        doc_text = text_data[doc_id]
        for start, end, label, entity_text in spans:
            total += 1
            if doc_text[start:end] != entity_text:
                bad += 1
    return total, bad

total1, bad1 = check_fold(text1, gold1)
print("fold1:", total1, "spans,", bad1, "mismatches")


fold1: 4120 spans, 0 mismatches


In [10]:
# Loading fold2 and running the same offset check on it
text2_path = hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_2_LNER/Train/fold2-text.json", repo_type="dataset")
gold2_path = hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_2_LNER/Train/fold2-gold.json", repo_type="dataset")

with open(text2_path) as f:
    text2 = json.load(f)

with open(gold2_path) as f:
    gold2 = json.load(f)

print(len(text2), "documents in fold2")

total2, bad2 = check_fold(text2, gold2)
print("fold2:", total2, "spans,", bad2, "mismatches")


35 documents in fold2
fold2: 4273 spans, 0 mismatches


In [11]:
# Getting a feel for document lengths and how often each label shows up
import statistics
from collections import Counter

all_text = {**text1, **text2}
all_gold = {**gold1, **gold2}

doc_lengths = [len(t) for t in all_text.values()]
print("num docs:", len(all_text))
print("char length - min/median/mean/max:",
      min(doc_lengths), statistics.median(doc_lengths),
      round(statistics.mean(doc_lengths)), max(doc_lengths))

label_counts = Counter()
for spans in all_gold.values():
    for start, end, label, entity_text in spans:
        label_counts[label] += 1

for label, count in label_counts.most_common():
    print(label, count)


num docs: 70
char length - min/median/mean/max: 10443 26620.0 31624 114044
DATE 1575
STAT 1260
PREC 1183
COURT 1013
AUTH 1003
CASENO 760
APP 407
RESP 381
JUDGE 245
A.COUNSEL 195
WIT 187
R.COUNSEL 184


In [12]:
# Mounting Drive to check the Phase 1 checkpoints are still there
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [13]:
# Listing what's inside the phase 1 checkpoint folder
import os

ckpt_root = "/content/drive/MyDrive/CS781/ckpt_p1"
for arch in os.listdir(ckpt_root):
    arch_path = os.path.join(ckpt_root, arch)
    print(arch, "->", os.listdir(arch_path))


decoder -> ['best_checkpoint.pt', 'training_log.csv']
encdec -> ['training_log.csv', 'best_checkpoint.pt']
moe -> ['training_log.csv', 'last_checkpoint.pt', 'best_checkpoint.pt']
convmix -> ['training_log.csv', 'last_checkpoint.pt', 'best_checkpoint.pt']


In [14]:
# Loading one checkpoint to make sure the file isn't corrupted and check what's inside it
import torch

ckpt = torch.load("/content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt", map_location="cpu")
print(type(ckpt))
print(ckpt.keys() if isinstance(ckpt, dict) else "not a dict")


<class 'dict'>
dict_keys(['model', 'optimizer', 'config', 'step', 'best_val_loss', 'args'])


In [15]:
# Checking the saved config to see the exact architecture settings used
print(ckpt['config'])
print("step:", ckpt['step'])
print("best_val_loss:", ckpt['best_val_loss'])


{'arch': 'decoder', 'vocab_size': 32000, 'block_size': 512, 'd_model': 768, 'n_layer': 12, 'n_head': 12, 'ffn_hidden': 2048, 'dropout': 0.0, 'n_enc_layer': 6, 'n_dec_layer': 6, 'enc_len': 768, 'dec_len': 256, 'n_experts': 8, 'top_k': 2, 'aux_loss_weight': 0.01, 'attn_every': 3, 'conv_kernel': 7, 'eot_id': 0, 'pad_id': 1, 'bos_id': 2}
step: 542
best_val_loss: 2.9732829585671423


In [16]:
# Checking the other three checkpoints load fine too
for arch in ["encdec", "moe", "convmix"]:
    c = torch.load(f"/content/drive/MyDrive/CS781/ckpt_p1/{arch}/best_checkpoint.pt", map_location="cpu")
    print(arch, "-> step:", c['step'], "| val_loss:", c['best_val_loss'], "| block_size:", c['config']['block_size'])


encdec -> step: 170 | val_loss: 4.9487828552722934 | block_size: 512
moe -> step: 170 | val_loss: 4.200264012813568 | block_size: 512
convmix -> step: 170 | val_loss: 3.9310310035943985 | block_size: 512


In [17]:
# Searching Drive for the saved tokenizer from phase 1
import subprocess
result = subprocess.run(["find", "/content/drive/MyDrive", "-iname", "tokenizer.json"], capture_output=True, text=True)
print(result.stdout)


/content/drive/MyDrive/CS781/data_p1/tokenizer.json



In [18]:
# Loading the phase 1 tokenizer and checking it gives us character offsets per token
from tokenizers import Tokenizer

tok = Tokenizer.from_file("/content/drive/MyDrive/CS781/data_p1/tokenizer.json")

sample_text = text1[doc_id][:200]
encoding = tok.encode(sample_text)

print(sample_text)
print()
for token, offset in zip(encoding.tokens[:20], encoding.offsets[:20]):
    print(token, offset, repr(sample_text[offset[0]:offset[1]]))


REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. 

REPORTABLE (0, 10) 'REPORTABLE'
ĠIN (10, 13) ' IN'
ĠTHE (13, 17) ' THE'
ĠSUPREME (17, 25) ' SUPREME'
ĠCOURT (25, 31) ' COURT'
ĠOF (31, 34) ' OF'
ĠINDIA (34, 40) ' INDIA'
ĠCRIMINAL (40, 49) ' CRIMINAL'
ĠAPPELLATE (49, 59) ' APPELLATE'
ĠJURISDICTION (59, 72) ' JURISDICTION'
ĠCRIMINAL (72, 81) ' CRIMINAL'
ĠAPPEAL (81, 88) ' APPEAL'
ĠNO (88, 91) ' NO'
. (91, 92) '.'
Ġ92 (92, 95) ' 92'
/ (95, 96) '/'
2015 (96, 100) '2015'
ĠJAG (100, 104) ' JAG'
E (104, 105) 'E'
ĠRAM (105, 109) ' RAM'


In [19]:
# Trying to get back the same characters after going from text to tokens and back
encoding_full = tok.encode(text1[doc_id])

test_span = gold1[doc_id][0]
start, end, label, entity_text = test_span
print("gold span:", start, end, label, entity_text)

overlapping = []
for i in range(len(encoding_full.offsets)):
    tok_start, tok_end = encoding_full.offsets[i]
    if tok_start < end and tok_end > start and not (tok_start == 0 and tok_end == 0):
        overlapping.append(i)

print("overlapping token indices:", overlapping)
print("overlapping tokens:", [encoding_full.tokens[i] for i in overlapping])

recovered_start = encoding_full.offsets[overlapping[0]][0]
recovered_end = encoding_full.offsets[overlapping[-1]][1]
print("recovered span:", recovered_start, recovered_end)
print("recovered text:", repr(text1[doc_id][recovered_start:recovered_end]))


gold span: 137 153 RESP STATE OF HARYANA
overlapping token indices: [26, 27, 28, 29, 30]
overlapping tokens: ['ĠSTATE', 'ĠOF', 'ĠH', 'ARY', 'ANA']
recovered span: 136 153
recovered text: ' STATE OF HARYANA'


In [20]:
# The first token's offset included a leading space that the gold span does not, fixing that
recovered_start = encoding_full.offsets[overlapping[0]][0]
recovered_end = encoding_full.offsets[overlapping[-1]][1]

doc_text = text1[doc_id]
while recovered_start < recovered_end and doc_text[recovered_start] == ' ':
    recovered_start += 1

print("fixed recovered span:", recovered_start, recovered_end)
print("fixed recovered text:", repr(doc_text[recovered_start:recovered_end]))
print("matches gold exactly:", doc_text[recovered_start:recovered_end] == entity_text)


fixed recovered span: 137 153
fixed recovered text: 'STATE OF HARYANA'
matches gold exactly: True


In [22]:
# Redefining recover_span and testing speed on just one document first
def recover_span(doc_text, encoding, start, end):
    overlapping = []
    for i in range(len(encoding.offsets)):
        tok_start, tok_end = encoding.offsets[i]
        if tok_start < end and tok_end > start and not (tok_start == 0 and tok_end == 0):
            overlapping.append(i)

    if not overlapping:
        return None, None

    rec_start = encoding.offsets[overlapping[0]][0]
    rec_end = encoding.offsets[overlapping[-1]][1]

    while rec_start < rec_end and doc_text[rec_start] == ' ':
        rec_start += 1

    return rec_start, rec_end


all_text = {**text1, **text2}
all_gold = {**gold1, **gold2}

import time

doc_id_test = list(all_gold.keys())[0]
doc_text = all_text[doc_id_test]

t0 = time.time()
encoding = tok.encode(doc_text)
print("encode time:", time.time() - t0)

t0 = time.time()
for start, end, label, entity_text in all_gold[doc_id_test]:
    recover_span(doc_text, encoding, start, end)
print("span recovery time:", time.time() - t0, "for", len(all_gold[doc_id_test]), "spans")


encode time: 0.021451950073242188
span recovery time: 169.12854170799255 for 104 spans


In [23]:
# Converting offsets to a plain list once, instead of re-touching the tokenizer object for every span
offsets_list = list(encoding.offsets)

def recover_span_fast(doc_text, offsets_list, start, end):
    overlapping = []
    for i in range(len(offsets_list)):
        tok_start, tok_end = offsets_list[i]
        if tok_start < end and tok_end > start and not (tok_start == 0 and tok_end == 0):
            overlapping.append(i)

    if not overlapping:
        return None, None

    rec_start = offsets_list[overlapping[0]][0]
    rec_end = offsets_list[overlapping[-1]][1]

    while rec_start < rec_end and doc_text[rec_start] == ' ':
        rec_start += 1

    return rec_start, rec_end


import time
t0 = time.time()
for start, end, label, entity_text in all_gold[doc_id_test]:
    recover_span_fast(doc_text, offsets_list, start, end)
print("span recovery time (fast):", time.time() - t0, "for", len(all_gold[doc_id_test]), "spans")


span recovery time (fast): 0.03276944160461426 for 104 spans


In [24]:
# Running the full check across all documents now that span recovery is actually fast
total = 0
exact_match = 0
mismatch_examples = []

for doc_id, spans in all_gold.items():
    doc_text = all_text[doc_id]
    encoding = tok.encode(doc_text)
    offsets_list = list(encoding.offsets)

    for start, end, label, entity_text in spans:
        total += 1
        rec_start, rec_end = recover_span_fast(doc_text, offsets_list, start, end)
        if rec_start == start and rec_end == end:
            exact_match += 1
        else:
            if len(mismatch_examples) < 10:
                mismatch_examples.append((doc_id, start, end, label, entity_text, rec_start, rec_end))

print("total spans:", total)
print("exact matches:", exact_match)
print("mismatches:", total - exact_match)
print()
for ex in mismatch_examples:
    print(ex)


total spans: 8393
exact matches: 8237
mismatches: 156

('115651329', 10266, 10299, 'PREC', 'State of M.P. vs. Kashiram & Ors.', 10266, 10300)
('975074', 7995, 8085, 'PREC', 'Gambangi Appalaswamy Naldu and Others v. Behara Venkataramanayya Patro [AIR 1984- SC 1728]', 7995, 8086)
('736324', 1963, 1976, 'AUTH', 'Railway Board', 1963, 1977)
('736324', 11218, 11231, 'AUTH', 'Railway Board', 11218, 11232)
('736324', 12849, 12901, 'PREC', 'Shankarsan Dash Vs. Union of India [1991 (3) SCC 47]', 12849, 12902)
('736324', 14121, 14174, 'PREC', 'Jatendra Kumar vs. State of Punjab [(1985) 1 SCC 122]', 14121, 14175)
('1243363', 17853, 17865, 'STAT', 'Constitution', 17853, 17867)
('1243363', 25083, 25099, 'AUTH', 'State Government', 25083, 25100)
('1243363', 25450, 25466, 'AUTH', 'State Government', 25450, 25467)
('373587', 5927, 5979, 'PREC', 'Hukam Singh v. State of Rajasthan AIR (1977 SC 1063)', 5927, 5980)


In [25]:
# Looking closely at one mismatch to see what's happening at the token boundary
doc_id_check = '736324'
doc_text_check = all_text[doc_id_check]
encoding_check = tok.encode(doc_text_check)
offsets_check = list(encoding_check.offsets)

start, end = 1963, 1976
print("gold text:", repr(doc_text_check[start:end]))

for i, (tok_start, tok_end) in enumerate(offsets_check):
    if tok_start < end + 5 and tok_end > start - 5:
        print(i, encoding_check.tokens[i], (tok_start, tok_end), repr(doc_text_check[tok_start:tok_end]))


gold text: 'Railway Board'
477 Ġthe (1958, 1962) ' the'
478 ĠRailway (1962, 1970) ' Railway'
479 ĠBoards (1970, 1977) ' Boards'
480 Ġinstructions (1977, 1990) ' instructions'


## Converting gold spans into per-token BIO labels

Testing make_token_labels on one real document: turning each gold
character span into a B-LABEL / I-LABEL / O tag for every token, so the
model can be trained on a per-token classification task instead of raw
character spans.


In [27]:
doc_id_test = list(text1.keys())[0]
print("using doc_id:", doc_id_test)

doc_text_test = text1[doc_id_test]
encoding_test = tok.encode(doc_text_test)
token_labels = make_token_labels(doc_text_test, encoding_test, gold1[doc_id_test])

for token, label in zip(encoding_test.tokens[:40], token_labels[:40]):
    print(token, "->", label)


using doc_id: 115651329
REPORTABLE -> O
ĠIN -> O
ĠTHE -> O
ĠSUPREME -> O
ĠCOURT -> O
ĠOF -> O
ĠINDIA -> O
ĠCRIMINAL -> O
ĠAPPELLATE -> O
ĠJURISDICTION -> O
ĠCRIMINAL -> O
ĠAPPEAL -> O
ĠNO -> O
. -> O
Ġ92 -> O
/ -> O
2015 -> O
ĠJAG -> O
E -> O
ĠRAM -> O
Ġ& -> O
ĠORS -> O
. -> O
Ġ.. -> O
Appellants -> O
ĠVersus -> O
ĠSTATE -> B-RESP
ĠOF -> I-RESP
ĠH -> I-RESP
ARY -> I-RESP
ANA -> I-RESP
Ġ.. -> O
Respondent -> O
ĠJ -> O
ĠU -> O
ĠD -> O
ĠG -> O
ĠM -> O
ĠE -> O
ĠN -> O


## Testing chunking on one long document

Breaking a document's tokens into overlapping 512-token pieces (same
chunk_document function as in data_prep.py), so none of the Phase-1
models (trained at block_size 512) need to see more than 512 tokens
at once. Checking the chunk count and overlap look right before trusting
this on the real data_prep.py run.


In [28]:
def chunk_document(doc_id, tokens, offsets, labels, chunk_tokens, chunk_overlap):
    chunks = []
    step = chunk_tokens - chunk_overlap
    num_tokens = len(tokens)

    start_tok = 0
    while start_tok < num_tokens:
        end_tok = min(start_tok + chunk_tokens, num_tokens)
        chunk_offsets = offsets[start_tok:end_tok]
        chunks.append({
            "doc_id": doc_id,
            "token_ids": tokens[start_tok:end_tok],
            "labels": labels[start_tok:end_tok],
            "chunk_char_start": chunk_offsets[0][0],
            "chunk_char_end": chunk_offsets[-1][1],
        })
        if end_tok == num_tokens:
            break
        start_tok += step

    return chunks


doc_id_test = list(text1.keys())[0]
doc_text_test = text1[doc_id_test]
encoding_test = tok.encode(doc_text_test)
token_labels_test = make_token_labels(doc_text_test, encoding_test, gold1[doc_id_test])

chunks = chunk_document(doc_id_test, encoding_test.ids, list(encoding_test.offsets), token_labels_test, 512, 64)

print("total tokens in doc:", len(encoding_test.ids))
print("number of chunks:", len(chunks))
for c in chunks:
    print("chunk tokens:", len(c["token_ids"]), "| char range:", c["chunk_char_start"], "-", c["chunk_char_end"])


total tokens in doc: 4028
number of chunks: 9
chunk tokens: 512 | char range: 0 - 2242
chunk tokens: 512 | char range: 1941 - 4363
chunk tokens: 512 | char range: 4026 - 6697
chunk tokens: 512 | char range: 6422 - 8786
chunk tokens: 512 | char range: 8484 - 11044
chunk tokens: 512 | char range: 10703 - 13086
chunk tokens: 512 | char range: 12767 - 15279
chunk tokens: 512 | char range: 15000 - 17312
chunk tokens: 444 | char range: 16997 - 18616
